# 03 — Chunking and Document Loading

## Why this notebook exists

Notebook 02 ranked whole documents by embedding each one as a single vector. That worked because our documents are tiny and each is about one thing. Real documents aren't: a support page covers warranty *and* hours *and* spare-part depots, and a PDF manual runs for pages. Squash all of that into one vector and it blurs — a question about spare parts and a question about warranty both match the same averaged document, and specific details get washed out.

The fix is to break documents into smaller, focused **chunks** and embed those, so retrieval can return the exact passage that answers a question. But first we have to *get the documents in* — including a PDF, which needs real extraction. This notebook does both: load the corpus (markdown + a PDF, via `pypdf`) into text with `source`/`page` metadata, split it into well-sized overlapping chunks, and show chunk-level retrieval returning the precise passage — with a citation — instead of a whole averaged document.

Requires an `OPENAI_API_KEY` (we embed chunks at the end to show the payoff). Still no vector database — just NumPy.

## What you'll learn

- How to **load** a mixed corpus — markdown files and a **PDF** (with `pypdf`) — into a uniform list of text units carrying `source` and `page` metadata.
- Why a single embedding per document is too coarse, and why **chunking** improves retrieval granularity.
- Three **chunking strategies**: naive fixed-size character splits, **token-based** splits (`tiktoken`), and **recursive/structure-aware** splits (`langchain-text-splitters`) — and the role of **chunk overlap**.
- How to attach **metadata** (`source`, `page`, `chunk_id`) to every chunk so retrieved passages can be cited.
- How **chunk-level retrieval** returns the exact relevant passage — the fix for notebook 02's whole-document blur.

## 1. Setup

We reuse notebook 02's embedding toolkit — `embed`, `embed_many`, and `cosine` — re-declared inline so this notebook stands alone. We load `OPENAI_API_KEY` from the environment (and from a local `.env` if `python-dotenv` is installed). Document loading and chunking themselves need no API; the key is used only at the end (Section 6) to embed chunks and show the retrieval payoff.

In [1]:
import os

# Optional: load a local .env if python-dotenv is installed. Real env vars win.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Key guard ──────────────────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    print("=" * 60)
    print("OPENAI_API_KEY is not set.")
    print("=" * 60)
    print()
    print("This notebook embeds chunks at the end (Section 6) to show the")
    print("retrieval payoff. Loading and chunking themselves need no key.")
    print()
    print("Set it and restart the kernel:")
    print("  export OPENAI_API_KEY=sk-...")
    raise SystemExit("Set OPENAI_API_KEY and restart the kernel to continue.")

print("OPENAI_API_KEY set ✓")

OPENAI_API_KEY set ✓


In [2]:
import numpy as np
from openai import OpenAI

EMBED_MODEL = "text-embedding-3-small"

openai_client = OpenAI()  # reads OPENAI_API_KEY from the environment


def embed(text: str) -> np.ndarray:
    """Embed a single string into a NumPy vector."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=text)
    return np.array(resp.data[0].embedding, dtype=np.float32)


def embed_many(texts: list) -> np.ndarray:
    """Embed a list of strings in ONE API call; returns a (len(texts), dim) matrix."""
    resp = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype=np.float32)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors (higher = more similar)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Setup OK")
print(f"Embedding model: {EMBED_MODEL}")

Setup OK
Embedding model: text-embedding-3-small


## 2. Loading Documents

Retrieval needs raw text, but our corpus is mixed: markdown files and a PDF. We load both into one uniform shape — a list of dicts, each with `source` (filename), `page` (a 1-based page number for PDFs, or `None` for markdown), and `text`.

Markdown is just a file read. The PDF needs real extraction: `pypdf` opens it and gives us the text of each page separately, which is why PDF pages become separate units — page numbers are useful metadata for citing where an answer came from.

In [3]:
from pathlib import Path
from pypdf import PdfReader

# Resolve the data folder whether the kernel runs from rag/ or from the repo root.
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("rag/data")


def load_documents(data_dir: Path) -> list:
    """Load every .md and .pdf in data_dir into a uniform list of text units.

    Each unit is a dict: {"source": filename, "page": int|None, "text": str}.
    Markdown files become one unit (page=None); each PDF page becomes its own
    unit (page=1, 2, ...), so we can cite the page an answer came from.
    """
    docs = []
    for path in sorted(data_dir.glob("*.md")):
        docs.append({"source": path.name, "page": None, "text": path.read_text(encoding="utf-8")})
    for path in sorted(data_dir.glob("*.pdf")):
        reader = PdfReader(str(path))
        for i, page in enumerate(reader.pages):
            docs.append({"source": path.name, "page": i + 1, "text": page.extract_text() or ""})
    return docs


documents = load_documents(DATA_DIR)

# Fail clearly if nothing loaded, instead of a confusing error later.
if not documents:
    raise FileNotFoundError(
        f"No documents found in {DATA_DIR}/. Run notebook 01 (and notebook 03's "
        "PDF-authoring step) first."
    )

print(f"Loaded {len(documents)} document units (markdown files + PDF pages):\n")
for d in documents:
    loc = d["source"] if d["page"] is None else f"{d['source']} p.{d['page']}"
    print(f"  {loc:<34} {len(d['text']):>5} chars")

Loaded 7 document units (markdown files + PDF pages):

  company_overview.md                  588 chars
  product_porter_p1.md                 470 chars
  product_porter_p2.md                 406 chars
  product_porter_p3.md                 372 chars
  support.md                           604 chars
  porter_p2_field_guide.pdf p.1        407 chars
  porter_p2_field_guide.pdf p.2        388 chars


In [4]:
# The PDF is where loading does real work — show what pypdf extracted.
pdf_units = [d for d in documents if d["source"].endswith(".pdf")]
print(f"The PDF contributed {len(pdf_units)} page-units.\n")
print("Page 1 extracted text:")
print("-" * 60)
print(pdf_units[0]["text"])

The PDF contributed 2 page-units.

Page 1 extracted text:
------------------------------------------------------------
Porter P2 - Field Service Guide
Preventive Maintenance Schedule
Every 250 operating hours: inspect the drive wheels for wear and
clean the lidar lenses with a microfiber cloth.
Every 500 operating hours: replace the air filter and check the
battery contactor torque (specification: 8 Nm).
Every 1,000 operating hours: perform a full calibration of the
floor-marker camera and update the navigation firmware.


## 3. Why Chunk?

Now that documents are loaded, why not just embed each one whole, like notebook 02 did? Two reasons:

1. **Granularity.** `support.md` covers warranty, support hours, *and* spare-part depots. As one vector it's an average of all three topics, so it matches every related question only weakly and never strongly. Split it into a warranty chunk, an hours chunk, and a depots chunk, and a question about spare parts matches the depots chunk *precisely*.
2. **Length.** Embedding models have an input limit and lose fidelity on long text. A 30-page PDF cannot be one useful vector. Chunks keep each embedded unit short and focused.

So we split documents into **chunks**: small spans of text, each embedded on its own. The art is choosing *where* to split. Too large and you're back to the blurring problem; too small and a chunk loses the context needed to make sense. The next section compares strategies.

> **Gotcha:** A good chunk is semantically self-contained — it should still make sense on its own when a model reads it in isolation. Splitting blindly every N characters often cuts a sentence (or a number and its unit) in half, which is why structure-aware splitting and overlap matter.

## 4. Chunking Strategies

We'll compare three ways to split text, using `support.md` as the sample:

1. **Naive fixed-size** — slice every N characters. Simple, but it cuts blindly through words and sentences.
2. **Recursive / structure-aware** — `RecursiveCharacterTextSplitter` from `langchain-text-splitters` tries to split on paragraph, then sentence, then word boundaries, so chunks end at natural breaks.
3. **Token-based** — split by *token* count (what the model and the embedding API actually measure) rather than characters, using a `tiktoken` encoder.

We also show **overlap**: letting consecutive chunks share a little text so a fact that straddles a boundary isn't lost.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample = next(d for d in documents if d["source"] == "support.md")["text"]
print(f"Sample: support.md ({len(sample)} chars)\n")

# 1. Naive fixed-size character chunks
def fixed_chunks(text: str, size: int = 200) -> list:
    return [text[i:i + size] for i in range(0, len(text), size)]

naive = fixed_chunks(sample, 200)
print(f"Naive fixed 200-char chunks: {len(naive)}")
print(f"  chunk 0 ends: ...{naive[0][-45:]!r}")
print("  ^ note how it can cut mid-word / mid-sentence\n")

# 2. Recursive / structure-aware chunks (respects paragraph & sentence breaks)
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=40)
recursive = splitter.split_text(sample)
print(f"Recursive 200-char chunks (overlap 40): {len(recursive)}")
print(f"  chunk 0 ends: ...{recursive[0][-45:]!r}")
print("  ^ ends on a natural boundary")

Sample: support.md (604 chars)

Naive fixed 200-char chunks: 4
  chunk 0 ends: ...'ive years is available for an additional 15% '
  ^ note how it can cut mid-word / mid-sentence

Recursive 200-char chunks (overlap 40): 5
  chunk 0 ends: ...'# Halcyon Robotics — Support & Warranty'
  ^ ends on a natural boundary


In [6]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")

# 3. Token-based splitting: chunk by token count, not character count.
tok_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=60, chunk_overlap=15
)
tok_chunks = tok_splitter.split_text(sample)
print(f"Token-based chunks (~60 tokens, overlap 15): {len(tok_chunks)}")
for i, c in enumerate(tok_chunks):
    print(f"  chunk {i}: {len(enc.encode(c)):>3} tokens, {len(c):>3} chars")

# Overlap in action: trailing text of one chunk reappears at the start of the next.
if len(recursive) > 1:
    print("\nOverlap carries context across the boundary:")
    print(f"  end of recursive chunk 0  : ...{recursive[0][-40:]!r}")
    print(f"  start of recursive chunk 1: {recursive[1][:40]!r}...")

Token-based chunks (~60 tokens, overlap 15): 3
  chunk 0:  46 tokens, 218 chars
  chunk 1:  48 tokens, 210 chars
  chunk 2:  35 tokens, 171 chars

Overlap carries context across the boundary:
  end of recursive chunk 0  : ...'# Halcyon Robotics — Support & Warranty'
  start of recursive chunk 1: 'All Porter robots ship with a standard t'...


## 5. Chunks With Metadata

Now we chunk the *whole* corpus into the canonical list we'll embed and search. Each chunk is a dict carrying the metadata we'll need later to cite answers: a unique `chunk_id`, its `source` document, its `page` (for PDF chunks), and the `text`. This list — chunks plus metadata — is exactly what the vector store in notebook 04 will hold.

We use the recursive splitter with a slightly larger window (300 chars, 50 overlap) so each chunk is a coherent passage.

In [7]:
from collections import Counter


def chunk_documents(documents: list, chunk_size: int = 300, chunk_overlap: int = 50) -> list:
    """Split every document unit into chunk records carrying metadata.

    Returns a list of dicts: {"chunk_id", "source", "page", "text"}.
    """
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = []
    cid = 0
    for d in documents:
        for piece in splitter.split_text(d["text"]):
            chunks.append({
                "chunk_id": cid,
                "source": d["source"],
                "page": d["page"],
                "text": piece,
            })
            cid += 1
    return chunks


chunks = chunk_documents(documents)
print(f"{len(documents)} document units → {len(chunks)} chunks\n")

print("First 3 chunk records:")
for ch in chunks[:3]:
    loc = ch["source"] if ch["page"] is None else f"{ch['source']} p.{ch['page']}"
    print(f"  chunk {ch['chunk_id']:>2} | {loc:<30} | {ch['text'][:48]!r}...")

print("\nChunks per source:")
for src, n in sorted(Counter(c["source"] for c in chunks).items()):
    print(f"  {src:<32} {n}")

7 document units → 16 chunks

First 3 chunk records:
  chunk  0 | company_overview.md            | '# Halcyon Robotics — Company Overview\n\nHalcyon R'...
  chunk  1 | company_overview.md            | 'Halcyon was founded by Dr. Priya Nair and Marcus'...
  chunk  2 | company_overview.md            | 'The company\'s mission is "to make warehouses saf'...

Chunks per source:
  company_overview.md              3
  porter_p2_field_guide.pdf        4
  product_porter_p1.md             2
  product_porter_p2.md             2
  product_porter_p3.md             2
  support.md                       3


## 6. Chunk-Level Retrieval Beats Document-Level

Here's the payoff. We embed every chunk (one batched call), then search the same way notebook 02 searched documents — but now the unit returned is a focused **passage** with a citation, not a whole averaged document. Ask about a PDF-only error code and retrieval pinpoints the exact page; ask about spare parts and it returns the depots passage from `support.md`, not the entire support document.

In [8]:
chunk_texts = [c["text"] for c in chunks]
chunk_matrix = embed_many(chunk_texts)
print(f"Embedded {len(chunks)} chunks into {chunk_matrix.shape}.\n")


def search_chunks(query: str, top_k: int = 2) -> list:
    """Return the top_k (chunk, score) pairs most similar to the query."""
    q = embed(query)
    scored = [(i, cosine(q, chunk_matrix[i])) for i in range(len(chunks))]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return [(chunks[i], s) for i, s in scored[:top_k]]


for query in [
    "What should I do about error code E204?",
    "Where are spare parts stocked?",
]:
    print(f"Q: {query}")
    for ch, score in search_chunks(query, top_k=2):
        loc = ch["source"] if ch["page"] is None else f"{ch['source']} p.{ch['page']}"
        snippet = " ".join(ch["text"].split())[:80]
        print(f"   {score:.3f}  [{loc}]  {snippet!r}...")
    print()

Embedded 16 chunks into (16, 1536).

Q: What should I do about error code E204?


   0.498  [porter_p2_field_guide.pdf p.2]  'Porter P2 - Field Service Guide Error Codes E101: drive motor over-temperature. '...
   0.363  [porter_p2_field_guide.pdf p.2]  'battery; if the fault persists, replace the pack. For all unresolved faults, con'...

Q: Where are spare parts stocked?


   0.636  [support.md]  'Spare parts are stocked in regional depots in Portland, Dallas, and Rotterdam. S'...
   0.335  [porter_p2_field_guide.pdf p.1]  'Porter P2 - Field Service Guide Preventive Maintenance Schedule Every 250 operat'...



## What you just learned

- **Loading** a mixed corpus into a uniform shape: markdown files plus a **PDF** read with `pypdf`, each unit carrying `source` and `page` metadata.
- Why one vector per document is too coarse — **chunking** restores granularity and keeps each embedded unit short enough to be faithful.
- Three **chunking strategies**: naive fixed-size (cuts blindly), **recursive/structure-aware** (splits on natural boundaries), and **token-based** (`tiktoken`) — plus **overlap** to preserve context across boundaries.
- Building **chunk records with metadata** (`chunk_id`, `source`, `page`, `text`) — the unit a vector store holds and retrieval returns.
- **Chunk-level retrieval** returns the exact relevant passage *with a citation*, fixing notebook 02's whole-document blur.

## What's missing

We're embedding every chunk and scanning all of them with a NumPy loop on each query. That's fine for a few dozen chunks, but it's a brute-force linear scan: at thousands or millions of chunks, embedding everything on the fly and comparing against every vector per query is far too slow, and we throw the embeddings away when the kernel restarts.

**Notebook 04 — `04_vector_store_with_faiss.ipynb`** introduces a real **vector store**: we build a FAISS index over the chunk embeddings once, query it in milliseconds with approximate nearest-neighbor search, keep the chunk metadata alongside it, and **persist** the index to disk so ingestion is a one-time cost. That's the last piece before we assemble the full retrieve-augment-generate pipeline in notebook 05.